# Lab 13 · A2A: delegate work to another agent

**Prepared by Mustafa Assaf** · Self-study: Go deeper · 15–20 minute lab

## Delegate a translation across an agent boundary

**New:** A2A lets a client discover an agent's capabilities from an **Agent Card**, then exchange messages through a protocol. Our remote function is deterministic: agents do not have to call an LLM.

The domain example is an **anti-corruption layer (ACL)**. A synthetic Stripe-style payload uses a provider's field names and minor currency units. The receiving agent translates it into ShopEasy's internal Payment model. This is not the Stripe API, and the local A2A transport opens no public server. The point is to protect the internal model, not spread the provider schema everywhere.

**At a glance**

![Lab 13 at a glance: Provider payload (synthetic payment), then Discover Agent Card (official A2A SDK), then Payment adapter (validate and translate), then valid?, then Internal Payment (Money value + order ID).](https://mkassaf.github.io/agentic-ai-course/assets/diagrams/13.svg)

## Think about the example

A payment provider and ShopEasy may use different field names and representations. An anti-corruption layer translates between them; A2A describes how agents discover and exchange messages.

**Concepts:** Anti-corruption layer; Published internal contract; Agent Card discovery.

Use the scenario above to consider these questions before running the code:

1. Why should Order not depend on the payment provider's field names?
2. Which part is the A2A protocol, and which part is domain translation?

## The idea in context

Delegation exchanges a task through an agent interface. This deterministic local agent demonstrates translation without a live payment service or model key.

A2A is an agent protocol; an anti-corruption layer protects the domain model. One does not imply the other.

| Key term | Meaning |
| --- | --- |
| **Anti-corruption layer** | Translation boundary that protects an internal domain model from a foreign one. |
| **Agent Card** | Discoverable description of an agent and its capabilities. |
| **Money value** | Amount and currency represented together in the internal model. |

## Pattern · Registry & Adapter

Discover another agent's capability and communicate through its published interface. The receiving agent acts as an ACL, adapting a foreign representation into a small internal contract. A2A discovery is agent discovery, not a replacement for microservice runtime service discovery.

## Setup
Run this cell once to load the course helpers, then run the steps below in order. This lab needs no model key.

In [ ]:
from importlib.util import find_spec
if not all(find_spec(name) for name in ['coursekit', 'a2a']):
    %pip install -q "agentic-ai-coursekit[a2a] @ https://github.com/mkassaf/agentic-ai-course/archive/9c78f2181b0f793c60b2b081c79ae74276026e7d.zip"

## Step 1 · Define the foreign payload and the internal model

`amount_received` is an integer number of minor units, not a floating-point price. The translator preserves 4200 cents as 4200 minor units and normalizes EUR. A malformed payload returns an error without claiming a payment succeeded.

The adapter takes JSON text and returns JSON text. First it validates the provider fields, then builds the internal Payment record. A bad input becomes an error result.

In [ ]:
import json
provider_payment = {"id": "pi_101", "amount_received": 4200, "currency": "eur", "status": "succeeded",
                    "metadata": {"order_id": "101"}}

def payment_adapter(text):
    try:
        payment = json.loads(text)
        if not isinstance(payment, dict):
            raise ValueError("Expected a payment object")
        p = payment  # use the provider's field names until translation below
        if type(p.get("amount_received")) is not int:
            raise ValueError("amount_received must be integer minor units")
        if p["amount_received"] < 0 or p.get("status") != "succeeded" or p.get("currency") not in ("eur", "usd"):
            raise ValueError("Expect a completed nonnegative payment and a currency")
        if not isinstance(p.get("id"), str) or not isinstance(p.get("metadata"), dict) or not isinstance(p["metadata"].get("order_id"), str):
            raise ValueError("order_id must be text")
        internal = {"order_id": p["metadata"]["order_id"], "payment_id": p["id"],
                    "total": {"minor_units": p["amount_received"], "currency": p["currency"].upper()}}
        return json.dumps(internal)
    except (ValueError, TypeError, KeyError):
        return json.dumps({"error": "Cannot translate this provider payment"})

print(payment_adapter(json.dumps(provider_payment)))

## Step 2 · Discover the translator and send a message

The helper uses the official A2A SDK to publish the Agent Card and exchange JSON-RPC messages over an in-process HTTP transport. The client discovers the card and sends text containing JSON; only the adapter knows the provider's field names. The helper is course code, not an A2A SDK method.

The helper starts an in-process A2A server and discovers its Agent Card. No real payment is sent or charged.

In [ ]:
from coursekit.protocol_examples import talk_over_a2a

ADAPTER = dict(name="Payment ACL", description="Translate a provider payment into the ShopEasy Payment model.",
               skill="Normalize payment")
card, reply = await talk_over_a2a(payment_adapter, json.dumps(provider_payment), **ADAPTER)
print("Discovered:", card["name"], card["skills"])
payment = json.loads(reply)
print("Internal payment:", payment)
assert payment["total"] == {"minor_units": 4200, "currency": "EUR"}
assert "amount_received" not in payment

## Step 3 · Keep provider changes outside Order

Order consumes the internal contract, not the foreign payload. Bad inputs should be rejected at the boundary; a failed translation cannot invent a total or mark an order paid. Test a string or boolean amount, malformed currencies, a pending payment, and a non-object JSON value. This classroom adapter supports only eur and usd; a real adapter needs an agreed currency catalog and minor-unit rules.

In [ ]:
bad_payloads = [{**provider_payment, "amount_received": value} for value in ("4200", True)]
bad_payloads += [{**provider_payment, "currency": value} for value in (None, "", 123)]
bad_payloads += [
                {**provider_payment, "status": "pending"}, []]
for payload in bad_payloads:
    _, reply = await talk_over_a2a(payment_adapter, json.dumps(payload), **ADAPTER)
    result = json.loads(reply)
    print(result)
    assert "error" in result

order = {"id": payment["order_id"], "paid_total": payment["total"]}
print("Order sees:", order)
assert order["paid_total"]["minor_units"] == 4200

## What you should see

The discovered agent is `Payment ACL`. A valid provider message becomes an internal Payment with order ID 101 and total 4200 EUR minor units. Seven invalid payloads return errors. Order sees the internal names, not `amount_received`.

## Try it

1. Add a provider field such as `receipt_url`. Does Order need to change?
2. Change the valid amount to 72. Should that mean 72 euros or 72 cents? Which contract says so?
3. Add a second provider adapter that returns the same internal Payment. Which Order code changes?
4. **Use it in your council.** Put an existing review capability behind an Agent Card. Specify its input, output, and error contract before connecting the chair.

## Takeaway

A2A provides discovery and message exchange. An ACL protects a domain model by translating at its boundary. Neither protocol discovery nor JSON alone proves the business data is valid; boundary checks still matter.